        # EWC 2026 Dota 2 fact-agent, clean version

        Этот ноутбук — короткая рабочая версия поверх одной SQLite-базы:

        - официальные ники и позиции из Liquipedia;
        - fantasy-очки по каждой карте;
        - role-category score: `core_avg`, `mid`, `support_avg`;
        - reliability-v2 score 1-100;
        - конструктор пользовательских fantasy-баннеров;
        - source-first агент, который не выдумывает внешние факты.

        Основной файл базы: `data/ewc_2026_fantasy_compact.sqlite or data/db/ewc_2026_fantasy_compact.sqlite` внутри проекта.
        


In [ ]:
from pathlib import Path
import os
import sys
import sqlite3
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().resolve().name == "notebooks" else Path.cwd().resolve()
DB_CANDIDATES = [
    PROJECT_ROOT / "data" / "ewc_2026_fantasy_compact.sqlite",
    PROJECT_ROOT / "data" / "db" / "ewc_2026_fantasy_compact.sqlite",
]
DB_PATH = next((path for path in DB_CANDIDATES if path.exists()), DB_CANDIDATES[0])
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from ewc_fact_agent_tools import (
    EWCFactAgent,
    ask,
    chat,
    db_status,
    roster,
    top_fantasy_maps,
    player_maps,
    role_map_summary,
    reliable_players_v2,
    reliable_role_slots_v2,
    reliability_backtest_v2,
    ti_qualified_teams,
    source_cache_status,
    banner_optimizer_players,
    banner_optimizer_role_slots,
    scoring_formula,
    source_urls,
    explain_sql_plan,
    explain_system_short,
)

agent = EWCFactAgent(DB_PATH)

def ask_v2(question: str, max_rows: int | None = None, use_llm: bool = False):
    """Main helper: returns AgentResult and displays markdown answer."""
    result = agent.ask(question, max_rows=max_rows, use_llm=use_llm)
    print(result.answer_markdown)
    return result

print("Clean EWC 2026 fact-agent готов.")
print(explain_system_short())


        ## 1. Быстрая проверка базы

        Если все ключевые объекты существуют, можно пользоваться агентом.
        


In [ ]:
display(db_status())


        ## 2. Основной агент

        Агент сначала пытается решить вопрос через SQLite. Если в вопросе есть внешний фильтр вроде `TI 2026 qualification`, он не делает вид, что этот список есть в базе, а просит сверить источник.
        


In [ ]:
examples = [
    "какой был состав у BetBoom?",
    "Укажи топ 15 лучших фентези игроков 1 позиции, их команды и лучшие фэнтези результаты",
    "Укажи топ 15 лучших фентези игроков 1 позиции из команд отобравшихся на TI 2026",
    "оптимизируй баннер для игроков 1 позиции из команд отобравшихся на TI 2026",
    "покажи надежных игроков для фэнтези",
    "покажи надежных саппортов для фэнтези",
    "самые надежные core пары",
    "покажи backtest модели надежности",
    "подробно объясни как считались fantasy очки",
]

for q in examples[:3]:
    print("\nQUESTION:", q)
    _ = ask_v2(q, max_rows=8)


        ## 3. Прямые SQL-friendly helpers

        Эти функции удобны, когда не нужен natural language router.
        


In [ ]:
display(roster("Team Falcons"))
display(top_fantasy_maps(position=1, limit=10))
display(top_fantasy_maps(position=1, ti2026_only=True, limit=10))
display(reliable_players_v2(position=1, limit=10))
display(reliable_role_slots_v2(role_slot="core_pair", limit=10))
display(ti_qualified_teams())
display(reliability_backtest_v2())


        ## 4. Fantasy banner optimizer

        Optimizer использует текущий fantasy-профиль и оценивает привлекательность пика по повторяемому потолку, а не по простой средней карте.

        По умолчанию саппорты исключены из рекомендаций, потому что их статистика в этой базе low-confidence.
        


In [ ]:
display(banner_optimizer_players(position=1, ti2026_only=True, limit=15))
display(banner_optimizer_role_slots(role_slot="core_pair", ti2026_only=True, limit=10))

# Natural language route:
_ = ask_v2("оптимизируй баннер для игроков 1 позиции из команд отобравшихся на TI 2026", max_rows=10)


        ## 5. Конструктор fantasy-баннеров

        Можно создать профиль под любые выпавшие коэффициенты. Профиль сохранится в тех же таблицах:

        - `fantasy_scoring_profiles`;
        - `fantasy_scoring_profile_stats`;
        - `fantasy_scoring_profile_banners`;
        - `fantasy_player_map_scores`;
        - `fantasy_team_role_map_scores`;
        - `fantasy_pick_value`.

        Ячейка ниже безопасная: пример не запускается автоматически.
        


In [ ]:
from fantasy_profile_constructor import create_or_replace_banner_profile

MY_CUSTOM_BANNER = {
    "core": [
        ("kills", 2.5),
        ("creep_score", 2.5),
        ("teamfight_participation", 1.8),
    ],
    "mid": [
        ("creep_score", 2.7),
        ("runes_grabbed", 1.8),
        ("teamfight_participation", 2.7),
    ],
    "support": [
        ("lotus", 3.2),
        ("watchers_taken", 2.1),
        ("teamfight_participation", 1.5),
    ],
}

# Чтобы реально создать профиль, поменяй False на True.
if False:
    con = sqlite3.connect(DB_PATH)
    profile_id = create_or_replace_banner_profile(
        con,
        "my_new_banner_profile",
        MY_CUSTOM_BANNER,
        profile_name="My new fantasy banner",
        set_default=False,  # True сделает профиль дефолтным
    )
    con.close()
    print("created:", profile_id)


        ## 6. Web/source tools

        Source-cache уже хранит OpenDota heroes и TI 2026 qualified teams. Для новых внешних фактов агент все равно не должен фантазировать: сначала источник, потом SQL-фильтр.
        


In [ ]:
display(source_cache_status())
display(ti_qualified_teams())
display(source_urls("команды отобравшиеся на TI 2026"))

# Если среда разрешает интернет, можно вручную попробовать:
# from ewc_fact_agent_tools import fetch_url_text
# text = fetch_url_text("https://liquipedia.net/dota2/The_International/2026")
# print(text[:1000])


        ## 7. SQL planner and confidence intervals

        `explain_sql_plan(...)` shows which deterministic route, views, filters and SQL template the agent will use. This is the fastest way to debug complex questions before letting GigaChat polish the answer.

        Reliability-v2 rows now include interval columns: `low_estimate`, `expected_estimate`, `high_estimate`, `uncertainty_score`, `confidence_label`.
        


In [ ]:
display(explain_sql_plan("top 15 fantasy pos1 players from TI 2026 qualified teams"))

cols = [
    "reliability_score_1_100",
    "official_name",
    "team_name",
    "official_position",
    "predicted_score_raw",
    "low_estimate",
    "expected_estimate",
    "high_estimate",
    "uncertainty_score",
    "confidence_label",
]
display(reliable_players_v2(position=1, ti2026_only=True, limit=10)[cols])


        ## 8. Dashboard and regression tests

        Dashboard is intentionally kept as a separate file so the notebook stays compact.

        - Dashboard file: `D:\test\test\ewc_fantasy_dashboard.py`
        - Tests file: `D:\test\test\regression_tests.py`
        


In [ ]:
DASHBOARD_PATH = DB_PATH.parent / "ewc_fantasy_dashboard.py"
TESTS_PATH = DB_PATH.parent / "regression_tests.py"

print("Dashboard:")
print(f"streamlit run {DASHBOARD_PATH}")

print("\nRegression tests:")
print(f"{sys.executable} {TESTS_PATH}")

# To run inside a notebook cell, uncomment:
# !python "D:\test\test\regression_tests.py"


        ## 9. Optional GigaChat post-processing

        По умолчанию ответы deterministic. Если в Colab/окружении есть `GIGACHAT_CREDENTIALS`, можно вызвать:

        ```python
        ask_v2("покажи надежных игроков для фэнтези", use_llm=True)
        ```

        LLM получает только черновик и таблицы, поэтому не должна добавлять новые числа вне данных.
        


In [ ]:
# Интерактивный режим:
# chat(use_llm=False)

# В конце работы можно закрыть соединение:
# agent.close()
